In [5]:
import sys
import platform

print("sys.executable:", sys.executable)
print("sys.version:", sys.version)
print("platform:", platform.platform())


sys.executable: /Users/owenhuang/.pyenv/versions/pythonfordata/bin/python
sys.version: 3.10.15 (main, Mar 14 2025, 20:54:33) [Clang 16.0.0 (clang-1600.0.26.6)]
platform: macOS-26.2-arm64-arm-64bit


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from linearmodels.panel import PanelOLS
from econml.dml import LinearDML
from pysyncon import Dataprep, Synth


In [7]:
import pandas as pd
from linearmodels.panel import PanelOLS

# sample panel data
df = pd.DataFrame({
    "entity": ["A","A","A","B","B","B"],
    "time": [1,2,3,1,2,3],
    "y": [10,12,13,8,9,11],
    "x": [1.0, 1.2, 1.4, 0.8, 1.0, 1.3]
})

df = df.set_index(["entity", "time"])

model = PanelOLS.from_formula("y ~ 1 + x + EntityEffects + TimeEffects", data=df)
res = model.fit(cov_type="clustered", cluster_entity=True)
print(res.summary)
# ...existing code...

                          PanelOLS Estimation Summary                           
Dep. Variable:                      y   R-squared:                        0.2500
Estimator:                   PanelOLS   R-squared (Between):              0.5867
No. Observations:                   6   R-squared (Within):               0.9107
Date:                Fri, Jan 30 2026   R-squared (Overall):              0.7595
Time:                        16:04:28   Log-likelihood                    1.0205
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.3333
Entities:                           2   P-value                           0.6667
Avg Obs:                       3.0000   Distribution:                     F(1,1)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-statistic (robust):          4.173e+28
                            

In [8]:
import pandas as pd
import numpy as np

# Load your data (assuming a CSV structure from Zillow/FRED)
# df = pd.read_csv('your_real_estate_data.csv')

# --- 1. SIMULATING SAMPLE DATA (Delete this block when using real data) ---
np.random.seed(42)
dates = pd.date_range(start='2018-01-01', end='2025-12-01', freq='MS')
cities = ['Austin', 'Boise', 'Phoenix', 'Chicago', 'Philadelphia', 'Boston', 'NYC']
data = []

for city in cities:
    for date in dates:
        # Simulate "Zoom Town" shock: Austin/Boise get huge inventory spike post-2023
        is_zoom_town = city in ['Austin', 'Boise', 'Phoenix']
        post_shock = date.year >= 2023
        
        # Base trends
        inventory = 3.0  # Base months of supply
        if is_zoom_town and post_shock:
            inventory += np.random.normal(2.5, 0.5) # The "Bust" Effect
        
        insurance_prem = 1200 + (200 * (date.year - 2018)) # Rising generally
        if is_zoom_town: insurance_prem *= 1.3 # Higher in boom areas
        
        data.append({
            'City': city,
            'Date': date,
            'Inventory_MoS': inventory,
            'Insurance_Premium': insurance_prem,
            'Mortgage_Rate': 6.5 + np.random.normal(0, 0.1), # National rate
            'Employment': np.random.randint(50000, 200000),
            'Is_Zoom_Town': 1 if is_zoom_town else 0
        })

df = pd.DataFrame(data)
# -------------------------------------------------------------------------

# --- 2. FEATURE ENGINEERING (The "Gap" & Log Transforms) ---
# Log Transform for Elasticity
df['ln_Inventory'] = np.log(df['Inventory_MoS'])
df['ln_Insurance'] = np.log(df['Insurance_Premium'])

# Create the Interaction Term for DiD (Zoom Town * Post-2023)
df['Post_2023'] = (df['Date'].dt.year >= 2023).astype(int)
df['Treat_Post'] = df['Is_Zoom_Town'] * df['Post_2023']

# Set Index for Panel Data (Entity = City, Time = Date)
df_panel = df.set_index(['City', 'Date'])

print("Data Prepared for Analysis:")
print(df_panel.head())

Data Prepared for Analysis:
                   Inventory_MoS  Insurance_Premium  Mortgage_Rate  \
City   Date                                                          
Austin 2018-01-01            3.0             1560.0       6.549671   
       2018-02-01            3.0             1560.0       6.486174   
       2018-03-01            3.0             1560.0       6.380219   
       2018-04-01            3.0             1560.0       6.714166   
       2018-05-01            3.0             1560.0       6.411477   

                   Employment  Is_Zoom_Town  ln_Inventory  ln_Insurance  \
City   Date                                                               
Austin 2018-01-01      169879             1      1.098612      7.352441   
       2018-02-01      160268             1      1.098612      7.352441   
       2018-03-01      137498             1      1.098612      7.352441   
       2018-04-01      162727             1      1.098612      7.352441   
       2018-05-01       91090  